## 0. Install dependencies

Run once. Safe to re-run.

In [ ]:
import sys, subprocess

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip_install(
    "google-genai",          # Gemini API (new unified SDK)
    "pandas",
    "python-dotenv",
    "pillow",
)
print("Dependencies installed.")


## 1. Imports, configuration, and API key

The Gemini API key is read from a `.env` file in the same folder as this notebook, as a line:

```
GEMINI_API_KEY=your-key-here
```

In [ ]:

import os
import re
import json
import time
import random
from pathlib import Path
from datetime import datetime

import pandas as pd
from PIL import Image
from IPython.display import display

from dotenv import load_dotenv
load_dotenv()  # expects GEMINI_API_KEY=... in a .env file next to this notebook

from google import genai
from google.genai import types as genai_types

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY not found. Create a .env file next to this notebook "
        "containing:\n  GEMINI_API_KEY=your-key-here"
    )

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini API key loaded from .env")


## 2. Configuration constants

Adjust for your document type.

In [ ]:

# --- Model & determinism -----------------------------------------------
GEMINI_MODEL_NAME = "gemini-3.1-flash-lite"
TEMPERATURE = 0  # deterministic output (ignored by gemini-3.6-flash)

# --- Retries & rate limiting (Gemini free tier) --------------------------
MAX_RETRIES = 3                 # automatic retries per image on invalid/malformed JSON responses
MAX_RATE_LIMIT_RETRIES = 12     # separate, larger budget for 429s — these are capacity waits, not bugs
RATE_LIMIT_MAX_SINGLE_WAIT_SECONDS = 600  # cap on a single wait even if Gemini suggests longer
REQUESTS_PER_MINUTE = 8         # conservative pacing to stay under free-tier RPM limits
MIN_SECONDS_BETWEEN_REQUESTS = 60.0 / REQUESTS_PER_MINUTE
BACKOFF_BASE_SECONDS = 5        # fallback exponential backoff, only used if Gemini's error
BACKOFF_MAX_SECONDS = 90        # doesn't include a retryDelay we can parse

# --- Expected record schema (order matters for CSV export) -----------------
SCHEMA_FIELDS = [
    "number_male",
    "number_female",
    "birth_month",
    "birth_day",
    "christ_day",
    "name",
    "parents_original",
    "godparents_original",
    "page_original",
]

# --- Fields the notebook forward-fills itself (blank in source = repeat) ---
INHERITED_FIELDS = ["birth_month"]

# --- Folders -----------------------------------------------------------
BASE_DIR = Path.cwd()
LOGS_DIR = BASE_DIR / "logs" / "metr_book"
CHECKPOINTS_DIR = BASE_DIR / "checkpoints_metr_book"
OUTPUT_DIR = BASE_DIR / "output" / "metr_book"

for d in (LOGS_DIR, CHECKPOINTS_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CHECKPOINT_CSV = CHECKPOINTS_DIR / "checkpoint.csv"
FINAL_CSV = OUTPUT_DIR / "metr_book_final.csv"

# --- Accepted image extensions ------------------------------------------
IMAGE_EXTENSIONS = {".jpg", ".jpeg"}

print("Configuration ready.")
print(f"Logs -> {LOGS_DIR}")
print(f"Checkpoint -> {CHECKPOINT_CSV}")
print(f"Final CSV -> {FINAL_CSV}")


## 3. Step 1 — Select the reference image and the folder of pages to transcribe

The reference image is a page you (a human) have already transcribed correctly, with a matching reference CSV. `INPUT_IMAGES_DIR` is the folder containing every other page image (JPEG) you want transcribed — the reference image itself is automatically excluded even if it lives in the same folder.

In [ ]:

REFERENCE_IMAGE_PATH = Path("images/metr_book/007767501_01087.jpeg")  # <-- EDIT THIS
INPUT_IMAGES_DIR = Path("images/metr_book")                            # <-- EDIT THIS
reference_csv_path = "metr_book_birth_ref.csv"                         # <-- EDIT THIS

REFERENCE_IMAGE_PATH = Path(str(REFERENCE_IMAGE_PATH).strip().strip('"').strip("'"))
if not REFERENCE_IMAGE_PATH.exists():
    raise FileNotFoundError(f"Reference image not found: {REFERENCE_IMAGE_PATH}")

RAW_REFERENCE_IMAGE_PATH = REFERENCE_IMAGE_PATH  # kept as the untouched original — Step 3 previews always use this

if not INPUT_IMAGES_DIR.exists() or not INPUT_IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"Input images folder not found: {INPUT_IMAGES_DIR}")

reference_csv_path = reference_csv_path.strip().strip('"').strip("'")
if not Path(reference_csv_path).exists():
    raise FileNotFoundError(f"Reference CSV not found: {reference_csv_path!r}")

print(f"Reference image: {REFERENCE_IMAGE_PATH}")
print(f"Input images folder: {INPUT_IMAGES_DIR}")
print(f"Reference CSV: {reference_csv_path}")


## 4. Step 2 — Gather the target images to transcribe

Sorted in **numeric order** (any digit run in the filename is compared as a number, not as text — so `_9` sorts before `_10`). The full ordered list is printed so you can see exactly which image any progress/failure message refers to; if a run stops partway, you can delete the already-processed images (everything up to and including the last one reported as `done`) before the next launch.

The reference image is excluded even if it's in this same folder.

In [ ]:

def _is_reference(p: Path) -> bool:
    try:
        return p.resolve() == REFERENCE_IMAGE_PATH.resolve()
    except OSError:
        return p.name == REFERENCE_IMAGE_PATH.name

def _natural_sort_key(p: Path):
    # Splits the filename on digit runs and compares those runs as integers,
    # so "page_9.jpg" sorts before "page_10.jpg" (plain string sort would not).
    parts = re.split(r"(\d+)", p.name)
    return [int(part) if part.isdigit() else part.lower() for part in parts]

target_image_paths = sorted(
    (p for p in INPUT_IMAGES_DIR.iterdir()
     if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS and not _is_reference(p)),
    key=_natural_sort_key,
)

if not target_image_paths:
    raise RuntimeError(f"No .jpg/.jpeg images found in {INPUT_IMAGES_DIR} (excluding the reference image).")

RAW_TARGET_IMAGE_PATHS = list(target_image_paths)  # kept as the untouched originals — Step 3 always processes from these

print(f"Found {len(target_image_paths)} image(s) to transcribe, in this order:")
for i, p in enumerate(target_image_paths, start=1):
    print(f"  {i:>4}. {p.name}")


## 5. Step 3 — Preprocess images: crop + downscale (automated)

Cuts token usage per request without manual per-image work.

- **`CROP_MARGIN_FRACTIONAL`**: an optional fixed crop, expressed as `(left, top, right, bottom)` — how much to **trim off each side**, as *fractions* of that image's width/height (`0.0`–`1.0`). E.g. trimming 3% off the left, top, and right, and nothing off the bottom, is `(0.03, 0.03, 0.03, 0.0)`. Works even if your scans aren't all exactly the same pixel size, since it's relative. Set to `None` to skip cropping. Use the preview cell below to dial it in before committing.
- **`MAX_IMAGE_DIMENSION_PX`**: after cropping, any image whose longest side still exceeds this is downscaled (high-quality Lanczos resampling) to it. Set to `None` to disable downscaling.

Processed copies are cached in a separate folder — your original scans are never modified. Re-running is cheap: already-processed files are skipped.

If your scans need *individual* per-image cropping instead (irregular framing that a single rectangle can't cover), see the note at the bottom of this section for fast manual-crop tools — do that pass *before* running this notebook, pointing `INPUT_IMAGES_DIR` at the cropped output folder.

In [ ]:

# --- Preprocessing configuration -----------------------------------------
# (left, top, right, bottom) — fraction of width/height trimmed off EACH side.
# Example: 3% off left/top/right, nothing off bottom:
CROP_MARGIN_FRACTIONAL = (0.1, 0.2, 0.23, 0.04)   #; None = no crop
MAX_IMAGE_DIMENSION_PX = 2200    # longest edge cap in pixels after cropping; None = no downscaling

# The cache folder name encodes the current crop/resize config, so changing
# CROP_MARGIN_FRACTIONAL or MAX_IMAGE_DIMENSION_PX always produces a FRESH
# set of processed files instead of silently reusing crops made with a
# different (possibly wrong) config from an earlier run.
import hashlib
_config_signature = hashlib.md5(f"{CROP_MARGIN_FRACTIONAL}|{MAX_IMAGE_DIMENSION_PX}".encode()).hexdigest()[:8]
PROCESSED_IMAGES_DIR = BASE_DIR / "images_processed" / "metr_book" / _config_signature
PROCESSED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

def _margins_to_box(w: int, h: int, margins) -> tuple[int, int, int, int]:
    left, top, right, bottom = margins
    box = (int(left * w), int(top * h), int(w - right * w), int(h - bottom * h))
    if box[0] >= box[2] or box[1] >= box[3]:
        raise ValueError(
            f"CROP_MARGIN_FRACTIONAL {margins} leaves an empty/inverted region "
            f"{box} for a {w}x{h} image. Each opposing pair must sum to < 1.0 "
            f"(e.g. left + right < 1.0, top + bottom < 1.0)."
        )
    return box


print(f"Crop margins (left, top, right, bottom): {CROP_MARGIN_FRACTIONAL}")
print(f"Max long-edge after crop: {MAX_IMAGE_DIMENSION_PX}")
print(f"Processed images cached in: {PROCESSED_IMAGES_DIR}")


### 5a. Preview the crop

Run this after setting `CROP_MARGIN_FRACTIONAL` above to see the resulting rectangle overlaid on the reference image, before it's applied to every page. Adjust the margins and re-run until it frames what you want kept.

This always previews `RAW_REFERENCE_IMAGE_PATH` — your original, untouched scan — so re-running this preview after Step 5b has already run once won't accidentally crop an already-cropped image.

In [ ]:

from PIL import ImageDraw

def preview_crop_box(image_path: Path, margins):
    img = Image.open(image_path).convert("RGB")
    if margins is None:
        display(img.resize((500, int(500 * img.height / img.width))))
        print("CROP_MARGIN_FRACTIONAL is None — no crop will be applied.")
        return
    w, h = img.size
    box_px = _margins_to_box(w, h, margins)
    preview = img.copy()
    ImageDraw.Draw(preview).rectangle(box_px, outline="red", width=max(3, w // 300))
    display(preview.resize((500, int(500 * preview.height / preview.width))))
    print(f"Crop box in pixels for this image: {box_px} (image size {w}x{h})")

preview_crop_box(RAW_REFERENCE_IMAGE_PATH, CROP_MARGIN_FRACTIONAL)


### 5b. Apply crop + downscale to every image

Reassigns `REFERENCE_IMAGE_PATH` and `target_image_paths` to point at the processed versions — nothing later in the notebook needs to change. Processes from `RAW_REFERENCE_IMAGE_PATH` and the original `target_image_paths`, never from an already-processed path, so re-running this cell (e.g. after changing the crop margins above) always starts fresh from the originals.

In [ ]:

def preprocess_image(src_path: Path) -> Path:
    out_path = PROCESSED_IMAGES_DIR / src_path.name
    if out_path.exists():
        return out_path  # already processed in a previous run

    img = Image.open(src_path).convert("RGB")
    changed = False

    if CROP_MARGIN_FRACTIONAL is not None:
        img = img.crop(_margins_to_box(img.width, img.height, CROP_MARGIN_FRACTIONAL))
        changed = True

    if MAX_IMAGE_DIMENSION_PX is not None and max(img.size) > MAX_IMAGE_DIMENSION_PX:
        scale = MAX_IMAGE_DIMENSION_PX / max(img.size)
        new_size = (max(1, int(img.width * scale)), max(1, int(img.height * scale)))
        img = img.resize(new_size, Image.LANCZOS)
        changed = True

    if not changed:
        return src_path  # nothing to do — use the original directly, no copy needed

    img.save(out_path, "JPEG", quality=92)
    return out_path


REFERENCE_IMAGE_PATH = preprocess_image(RAW_REFERENCE_IMAGE_PATH)
target_image_paths = [preprocess_image(p) for p in RAW_TARGET_IMAGE_PATHS]

print(f"Reference image (processed): {REFERENCE_IMAGE_PATH}")
print(f"Processed {len(target_image_paths)} target image(s).")


## 6. Step 4 — Automatic reference CSV → JSON conversion

Also forward-fills `INHERITED_FIELDS` in the reference itself, so the reference JSON shown to Gemini demonstrates the *filled* convention as a worked example, while target pages should leave those fields blank when blank — the notebook fills those in afterward.

In [ ]:

def csv_to_filled_records(csv_path: str) -> list[dict]:
    df = pd.read_csv(csv_path, dtype=str).fillna("")
    missing = [c for c in SCHEMA_FIELDS if c not in df.columns]
    if missing:
        raise ValueError(f"Reference CSV is missing expected columns: {missing}")
    df = df[SCHEMA_FIELDS]

    for col in INHERITED_FIELDS:
        df[col] = df[col].replace("", pd.NA).ffill().fillna("")

    return df.to_dict(orient="records")

REFERENCE_CSV_PATH = reference_csv_path
reference_records = csv_to_filled_records(REFERENCE_CSV_PATH)

REFERENCE_JSON_PATH = CHECKPOINTS_DIR / "reference.json"
with open(REFERENCE_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(reference_records, f, ensure_ascii=False, indent=2)

print(f"Converted {len(reference_records)} reference rows -> {REFERENCE_JSON_PATH}")
pd.DataFrame(reference_records)


## 7. Step 5 — Configure Gemini (deterministic)

In [ ]:

generation_config = genai_types.GenerateContentConfig(
    temperature=TEMPERATURE,
    response_mime_type="application/json",
)

print(f"Model ready: {GEMINI_MODEL_NAME} (temperature={TEMPERATURE})")


## 8. Step 6 — One-shot prompt template

Kept in a plain variable so the **Edit prompt** option later can modify it at runtime without re-running earlier cells.

In [ ]:

DOCUMENT_LANGUAGE_HINT = "a Slavic language written in Cyrillic script in Chernigiv, russian empire, end of 1800s"

DEFAULT_PROMPT_TEMPLATE = f'''You are an expert paleographer transcribing a scanned page from an old
handwritten church register in {DOCUMENT_LANGUAGE_HINT}.

You are given:
1. A REFERENCE page image (already transcribed by a human, names should be according to known names in russian empire of that time and place).
2. The REFERENCE page's correct JSON transcription (ground truth), which
   demonstrates the exact format, conventions, and handwriting style expected.
3. A NEW target page image that you must transcribe.

STRICT RULES:
- Transcribe the historical spelling EXACTLY as written. Do NOT modernize,
  correct, or normalize spelling, punctuation, or grammar in any way.
- If a letter, word, or number is illegible, smudged, or ambiguous,
  transcribe your best-guess reading and append "[?]" immediately after it
  (e.g. "Марі[?]я" or "45[?]"). Never silently omit uncertain content, and
  never invent content that is not visually present.
- Each row of the register describes one (rarely two when tweens) person.
- The "birth_month" field is typically written ONLY above the block of babies born in that month and should be written
  for the first entry and left blank for every other baby of the same month.
- The value for "page_original" field is taken from the handwritten text in the upper right corner of the image.
- The text about priests starting usually from "Священник" should be omitted. This text usually follows each record after
  "оба православные" (or rarely other confession of parents).
- Handwritten notes made chaotically on top of the usual records should be omitted.

OUTPUT FORMAT:
- Return ONLY a JSON array of objects, nothing else (no markdown fences,
  no commentary, no explanations before or after).
- Each object must have EXACTLY these 9 keys, in this order:
  {json.dumps(SCHEMA_FIELDS)}
- Preserve the top-to-bottom row order of the page.
'''

# Mutable at runtime (the "Edit prompt" workflow step reassigns this).
PROMPT_TEMPLATE = DEFAULT_PROMPT_TEMPLATE
print(PROMPT_TEMPLATE)


## 9. Step 7 — Gemini call: JSON validation, retries, rate limiting, logging

In [ ]:

class RateLimiter:
    def __init__(self, min_interval_seconds: float):
        self.min_interval = min_interval_seconds
        self._last_call = 0.0
        self._cooldown_until = 0.0  # monotonic timestamp; set after a 429 tells us to wait

    def wait(self):
        now = time.monotonic()
        target = max(self._last_call + self.min_interval, self._cooldown_until)
        remaining = target - now
        if remaining > 0:
            time.sleep(remaining)
        self._last_call = time.monotonic()

    def set_cooldown(self, seconds: float):
        # Applies to every subsequent call (this image's retries AND the next
        # image), so we don't immediately slam the same quota wall again.
        self._cooldown_until = max(self._cooldown_until, time.monotonic() + seconds)

rate_limiter = RateLimiter(MIN_SECONDS_BETWEEN_REQUESTS)


def _parse_retry_delay_seconds(e) -> float | None:
    # Gemini's 429 responses include a structured RetryInfo.retryDelay
    # (e.g. "37s") telling us exactly how long to wait. Try the structured
    # path first, then fall back to regex over the stringified error, since
    # error shapes vary across SDK versions.
    try:
        details = getattr(e, "details", None)
        if isinstance(details, dict):
            err_body = details.get("error", details)
            for d in err_body.get("details", []) or []:
                if "RetryInfo" in str(d.get("@type", "")) and "retryDelay" in d:
                    m = re.match(r"(\d+(?:\.\d+)?)", str(d["retryDelay"]).strip())
                    if m:
                        return float(m.group(1))
    except Exception:
        pass

    text = str(e)
    for pattern in (
        r"retryDelay['\"]?\s*[:=]\s*['\"]?(\d+(?:\.\d+)?)s",
        r"retry_delay\s*\{\s*seconds:\s*(\d+)",
        r"[Rr]etry in (\d+(?:\.\d+)?)s",
    ):
        m = re.search(pattern, text)
        if m:
            return float(m.group(1))
    return None

class InvalidTranscriptionError(Exception):
    pass


def _short(msg, limit=140) -> str:
    # Trim long exception text so log lines stay one line, not a wall of text.
    msg = str(msg).replace("\n", " ").strip()
    return msg if len(msg) <= limit else msg[:limit].rstrip() + "..."


def _validate_records(records) -> list[dict]:
    if not isinstance(records, list) or len(records) == 0:
        raise InvalidTranscriptionError("Response is not a non-empty JSON array.")
    cleaned = []
    for row in records:
        if not isinstance(row, dict):
            raise InvalidTranscriptionError(f"Row is not an object: {row!r}")
        missing = [k for k in SCHEMA_FIELDS if k not in row]
        if missing:
            raise InvalidTranscriptionError(f"Row missing keys {missing}: {row!r}")
        cleaned.append({k: ("" if row[k] is None else str(row[k])) for k in SCHEMA_FIELDS})
    return cleaned


def _extract_json_array(text: str):
    text = text.strip()
    # tolerate accidental markdown fences despite instructions
    text = re.sub(r"^```(json)?", "", text.strip(), flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text.strip()).strip()
    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1 or end < start:
        raise InvalidTranscriptionError("No JSON array found in response.")
    return json.loads(text[start:end + 1])


def call_gemini_for_image(image_id: str, image_path: Path, prompt_text: str, log_prefix: str = "") -> list[dict]:
    # Calls Gemini for a single image, with retries, logging, and validation.
    # Rate-limit (429) errors get their own generous retry budget and wait
    # exactly as long as Gemini's error tells us to (via retryDelay), rather
    # than guessing with exponential backoff. Invalid/malformed JSON
    # responses use a separate, smaller retry budget (MAX_RETRIES).
    prefix = f"{log_prefix} " if log_prefix else ""
    reference_image = Image.open(REFERENCE_IMAGE_PATH)
    with open(REFERENCE_JSON_PATH, "r", encoding="utf-8") as f:
        reference_json_text = f.read()
    target_image = Image.open(image_path)

    last_error = None
    json_attempts = 0
    rate_limit_attempts = 0
    log_attempt = 0

    while True:
        rate_limiter.wait()
        log_attempt += 1
        try:
            content = [
                prompt_text,
                "REFERENCE PAGE IMAGE:",
                reference_image,
                "REFERENCE PAGE — CORRECT JSON TRANSCRIPTION:",
                reference_json_text,
                "NEW TARGET PAGE IMAGE TO TRANSCRIBE:",
                target_image,
            ]
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL_NAME,
                contents=content,
                config=generation_config,
            )
            raw_text = response.text or ""

            log_path = LOGS_DIR / f"{image_id}.txt"
            with open(log_path, "a", encoding="utf-8") as f:
                f.write(f"--- attempt {log_attempt} @ {datetime.now().isoformat()} ---\n")
                f.write(raw_text)
                f.write("\n\n")

            records = _extract_json_array(raw_text)
            validated = _validate_records(records)
            return validated

        except Exception as e:
            last_error = e
            err_text = str(e).lower()
            is_rate_limit = "429" in err_text or "resource_exhausted" in err_text or "quota" in err_text

            if is_rate_limit:
                rate_limit_attempts += 1
                if rate_limit_attempts > MAX_RATE_LIMIT_RETRIES:
                    print(f"{prefix}{image_id}: giving up after {rate_limit_attempts - 1} "
                          f"rate-limit wait(s) ({_short(e)})")
                    break

                delay = _parse_retry_delay_seconds(e)
                if delay is not None:
                    delay = min(delay + 2, RATE_LIMIT_MAX_SINGLE_WAIT_SECONDS)  # small buffer, capped
                    source = "Gemini-suggested wait"
                else:
                    delay = min(BACKOFF_MAX_SECONDS, BACKOFF_BASE_SECONDS * (2 ** (rate_limit_attempts - 1)))
                    delay += random.uniform(0, 2)
                    source = "no retryDelay found, estimated backoff"

                rate_limiter.set_cooldown(delay)
                print(f"{prefix}{image_id}: rate-limited (wait {rate_limit_attempts}/{MAX_RATE_LIMIT_RETRIES}) — "
                      f"{source}: {delay:.0f}s")
                # No time.sleep() here — rate_limiter.wait() at the top of the
                # next loop iteration (for this image or the next one) enforces it.
                continue

            json_attempts += 1
            if json_attempts >= MAX_RETRIES:
                print(f"{prefix}{image_id}: FAILED after {json_attempts} attempts "
                      f"({_short(e)})")
                break
            print(f"{prefix}{image_id}: invalid response on attempt "
                  f"{json_attempts}/{MAX_RETRIES} ({_short(e)}) — retrying")

    raise InvalidTranscriptionError(f"{image_id} failed: {_short(last_error)}")

print("Gemini call function ready (validation + retries + logging + rate limiting).")


## 10. Step 8 — Inheriting fields across rows and images

A running `carry_state` remembers the last known value of each `INHERITED_FIELDS` column across the *entire* document (not just within one image), so a group that continues onto the next page still inherits correctly.

In [ ]:

def apply_inheritance(records: list[dict], carry_state: dict) -> list[dict]:
    filled = []
    for row in records:
        row = dict(row)
        for field in INHERITED_FIELDS:
            if row.get(field, "").strip() == "":
                row[field] = carry_state.get(field, "")
            else:
                carry_state[field] = row[field]
        filled.append(row)
    return filled

print("Inheritance function ready.")


## 11. Step 9 — Checkpointing & resume-from-checkpoint

In [ ]:

def load_checkpoint():
    # Returns (dataframe, processed_image_ids_set, carry_state).
    if CHECKPOINT_CSV.exists():
        df = pd.read_csv(CHECKPOINT_CSV, dtype=str).fillna("")
        if "_source_image" in df.columns:
            processed = set(df["_source_image"].unique())
        else:
            processed = set()
        carry_state = {}
        if len(df) > 0:
            for field in INHERITED_FIELDS:
                nonblank = df[df[field] != ""][field]
                carry_state[field] = nonblank.iloc[-1] if len(nonblank) else ""
        return df, processed, carry_state
    else:
        return pd.DataFrame(columns=SCHEMA_FIELDS + ["_source_image"]), set(), {}


def save_checkpoint(df: pd.DataFrame):
    df.to_csv(CHECKPOINT_CSV, index=False)


checkpoint_df, processed_images, carry_state = load_checkpoint()
print(f"Resumed checkpoint: {len(processed_images)} image(s) already processed: "
      f"{sorted(processed_images) if processed_images else 'none'}")
print(f"Carry state so far: {carry_state}")
checkpoint_df.tail()


## 12. Step 10 — Preview the first transcribed image

This transcribes the first image in `target_image_paths` and shows you the image + resulting table before committing to the full run.

In [ ]:

preview_image_path = target_image_paths[0]
preview_image_id = preview_image_path.stem

def run_preview(prompt_text: str):
    print(f"Transcribing preview image {preview_image_id} ...")
    records = call_gemini_for_image(preview_image_id, preview_image_path, prompt_text)
    local_carry = dict(carry_state)
    filled = apply_inheritance(records, local_carry)
    preview_df = pd.DataFrame(filled)
    return filled, preview_df

preview_records, preview_df = run_preview(PROMPT_TEMPLATE)

_preview_img = Image.open(preview_image_path)
display(_preview_img.resize((500, int(500 * _preview_img.height / _preview_img.width))))
display(preview_df)


## 13. Step 11 — Decide: Continue / Edit prompt / Retry / Quit

Run this cell after reviewing the preview above. It will keep asking until you choose **Continue** or **Quit**.

In [ ]:

QUIT_REQUESTED = False

def preview_decision_loop():
    global PROMPT_TEMPLATE, preview_records, preview_df, QUIT_REQUESTED
    while True:
        choice = input(
            "\nReview the preview above.\n"
            "  [c] Continue with the full run\n"
            "  [e] Edit prompt (then re-transcribes this preview image)\n"
            "  [r] Retry (re-transcribe this preview image, same prompt)\n"
            "  [q] Quit\n"
            "Choice [c/e/r/q]: "
        ).strip().lower()

        if choice == "c":
            print("Continuing to the full run.")
            return "continue"
        elif choice == "e":
            print("Paste the new prompt text, then press Enter twice to finish:")
            lines = []
            while True:
                line = input()
                if line == "" and lines and lines[-1] == "":
                    break
                lines.append(line)
            new_prompt = "\n".join(lines).rstrip()
            if new_prompt:
                PROMPT_TEMPLATE = new_prompt
            preview_records, preview_df = run_preview(PROMPT_TEMPLATE)
            display(preview_df)
        elif choice == "r":
            preview_records, preview_df = run_preview(PROMPT_TEMPLATE)
            display(preview_df)
        elif choice == "q":
            QUIT_REQUESTED = True
            print("Quitting. Run no further cells (checkpoint/final export are safe to skip).")
            return "quit"
        else:
            print("Please enter one of: c, e, r, q")

decision = preview_decision_loop()


## 14. Step 12 — Full run over all target images

Skips images already in the checkpoint (resume). Saves the checkpoint after every successfully processed image. Prints progress as both a percentage and an image count.

In [ ]:

if QUIT_REQUESTED:
    print("Quit was selected in the previous step — skipping the full run.")
else:
    all_target_ids = [p.stem for p in target_image_paths]
    remaining = [p for p in target_image_paths if p.stem not in processed_images]

    total = len(all_target_ids)
    already_done = total - len(remaining)

    def progress_prefix(done, total):
        pct = done * 100 // max(total, 1)
        width = len(str(total))
        return f"[{done:>{width}}/{total} | {pct:>3}%]"

    print(f"{progress_prefix(already_done, total)} starting "
          f"({len(remaining)} image(s) remaining, {already_done} already checkpointed)")

    rows_accum = checkpoint_df.to_dict(orient="records")

    # If the preview image was accepted (choice == "continue") and isn't
    # already checkpointed, seed it in so we don't re-call Gemini for it.
    if decision == "continue" and preview_image_id not in processed_images:
        for r in preview_records:
            row = dict(r)
            row["_source_image"] = preview_image_id
            rows_accum.append(row)
        for field in INHERITED_FIELDS:
            nonblank = [r[field] for r in preview_records if r.get(field, "") != ""]
            if nonblank:
                carry_state[field] = nonblank[-1]
        processed_images.add(preview_image_id)
        checkpoint_df = pd.DataFrame(rows_accum, columns=SCHEMA_FIELDS + ["_source_image"])
        save_checkpoint(checkpoint_df)
        remaining = [p for p in remaining if p.stem != preview_image_id]
        already_done += 1
        print(f"{progress_prefix(already_done, total)} {preview_image_id}: "
              f"done (from preview, {len(preview_records)} rows)")

    for image_path in remaining:
        image_id = image_path.stem
        prefix = progress_prefix(already_done, total)
        try:
            records = call_gemini_for_image(image_id, image_path, PROMPT_TEMPLATE, log_prefix=prefix)
        except InvalidTranscriptionError as e:
            print(f"{prefix} {image_id}: SKIPPED — {_short(e)}")
            continue

        filled = apply_inheritance(records, carry_state)
        for r in filled:
            row = dict(r)
            row["_source_image"] = image_id
            rows_accum.append(row)

        checkpoint_df = pd.DataFrame(rows_accum, columns=SCHEMA_FIELDS + ["_source_image"])
        save_checkpoint(checkpoint_df)
        processed_images.add(image_id)

        already_done += 1
        print(f"{progress_prefix(already_done, total)} {image_id}: done ({len(filled)} rows)")

    print("Full run complete." if not remaining else "Run finished with some images skipped — see messages above.")


## 15. Step 13 — Final CSV export

In [ ]:

final_df = pd.read_csv(CHECKPOINT_CSV, dtype=str).fillna("")
export_df = final_df[SCHEMA_FIELDS]  # drop internal _source_image column
export_df.to_csv(FINAL_CSV, encoding="utf-8-sig", index=False)

print(f"Exported {len(export_df)} rows -> {FINAL_CSV}")
export_df
